In [ ]:
import pandas as pd
import numpy as np
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss

# Load data
df = pd.read_csv('data/features.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

FEATURE_COLS = [
    'home_form_last5',
    'away_form_last5',
    'home_avg_scored',
    'home_avg_conceded',
    'away_avg_scored',
    'away_avg_conceded',
    'ref_matches',
    'ref_home_win_rate',
    'ref_yellows_avg',
    'ref_reds_avg',
    'ref_goals_avg',
    'elo_home',
    'elo_away',
    'elo_diff',
    'h2h_matches',
    'h2h_home_wins',
    'h2h_draws',
    'h2h_goals_avg',
]

X = df[FEATURE_COLS].values
y = df['target'].values

# Objective function for Optuna
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 2, 6),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'eval_metric':       'mlogloss',
        'random_state':      42,
    }

    tscv = TimeSeriesSplit(n_splits=5)
    fold_losses = []

    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = XGBClassifier(**params)
        model.fit(X_train, y_train, verbose=False)

        probs = model.predict_proba(X_val)
        fold_losses.append(log_loss(y_val, probs))

    return np.mean(fold_losses)

# Run optimization
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best log_loss: {study.best_value:.4f}")
print(f"✅ Best params: {study.best_params}")

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
import pickle

best_params = study.best_params
best_params['eval_metric'] = 'mlogloss'
best_params['random_state'] = 42

# Train final model with best params
final_model = XGBClassifier(**best_params)

# FIX: Use TimeSeriesSplit instead of random KFold (cv=3).
# Random KFold leaks future data into the calibration step.
calibrated_model = CalibratedClassifierCV(
    final_model,
    method='isotonic',
    cv=TimeSeriesSplit(n_splits=3)
)

calibrated_model.fit(X, y)

# Save replacing old model
with open('data/model.pkl', 'wb') as f:
    pickle.dump(calibrated_model, f)

# Quick sanity check
sample = X[-5:]
probs = calibrated_model.predict_proba(sample)
sample_df = df.tail(5)[['Date', 'HomeTeam', 'AwayTeam']].copy()
sample_df['actual'] = y[-5:]
sample_df['prob_home'] = probs[:, 0].round(3)
sample_df['prob_draw'] = probs[:, 1].round(3)
sample_df['prob_away'] = probs[:, 2].round(3)

print(sample_df.to_string())
print("\n✅ Tuned model saved to data/model.pkl")